# Expert Reviews Data Cleaning

In this notebook, I clean the Expert Reviews data and prepare the CRITICS and CRITIC_REVIEWS tables for our database.

I keep the code simple so I can understand and explain each step.


## 1. Import pandas

I use pandas to work with the review data.

In [7]:
# JP: I import pandas because I use it to work with the review data.
import pandas as pd


In [8]:
# JP: I tell Python to use my Downloads folder because that is where my Excel file is saved.
import os
os.chdir(os.path.expanduser("~/Downloads"))

## 2. Load the Expert Reviews data

I load the original Excel file before making any changes.

In [9]:
# JP: I load the original Expert Reviews Excel file into Python.
expert = pd.read_excel("ExpertReviewsClean43LIWC.xlsx")


In [10]:
# JP: I look at the first five rows so I can see what the data looks like.
expert.head()


,url,idvscore,reviewer,dateP,Rev,WC,Analytic,Clout,Authentic,Tone,...,Comma,Colon,SemiC,QMark,Exclam,Dash,Quote,Apostro,Parenth,OtherP
0,https://www.metacritic.com/movie/bronson,100.0,"""Andrew O'Hehir""",None,'Bronson owes a little or a lot to Kubrick s ...,25,73.88,11.52,6.70,25.77,...,4.00,0.00,0.0,0.0,0.0,0.00,0,8.00,0.0,0.0
1,https://www.metacritic.com/movie/bronson,90.0,'A.O. Scott',None,'Bronson invites you to admire its protagonis...,30,13.07,97.69,1.00,99.00,...,6.67,0.00,0.0,0.0,0.0,0.00,0,6.67,0.0,0.0
2,https://www.metacritic.com/movie/bronson,90.0,None,None,'Whether it s Peterson/Bronson s more theatri...,40,72.69,77.33,7.84,2.89,...,7.50,0.00,0.0,0.0,0.0,12.50,0,5.00,0.0,2.5
3,https://www.metacritic.com/movie/bronson,83.0,'Noel Murray',None,'There are two Bronsons on display here: the ...,39,65.46,99.00,1.00,25.77,...,5.13,2.56,0.0,0.0,0.0,0.00,0,5.13,0.0,0.0
4,https://www.metacritic.com/movie/bronson,80.0,'Joshua Rothkopf',None,'Refn has somehow found his way to an authent...,24,88.46,89.42,1.79,25.77,...,8.33,0.00,0.0,0.0,0.0,4.17,0,8.33,0.0,0.0


In [11]:
# JP: I check how many rows and columns are in the dataset.
expert.shape


(238973, 98)

## 3. Remove duplicate reviews

I remove exact duplicate rows so the same review is not counted more than once.

In [12]:
# JP: I check how many exact duplicate rows are in the dataset.
expert.duplicated().sum()


np.int64(27)

In [13]:
# JP: I remove duplicate rows because I do not want the same review to appear more than once.
expert = expert.drop_duplicates()


## 4. Remove reviews without review text

A review needs actual review text for the language variables to be useful.

In [14]:
# JP: I check how many reviews are missing their review text.
expert["Rev"].isna().sum()


np.int64(2)

In [15]:
# JP: I remove rows without review text because they are not useful for analysing reviews.
expert = expert.dropna(subset=["Rev"])


## 5. Remove placeholder reviews

Some rows use the text 'None' instead of an actual review.

In [16]:
# JP: I count reviews where the review text is the placeholder 'None'.
expert["Rev"].astype(str).str.strip().eq("'None'").sum()


np.int64(4555)

In [17]:
# JP: I remove reviews where the review text is only the placeholder 'None'.
expert = expert[expert["Rev"].astype(str).str.strip() != "'None'"]


## 6. Check the emotion variables

These are the emotion variables used in our analysis.

In [18]:
# JP: I check the positive emotion, negative emotion, and tone variables used in our analysis.
expert[["posemo", "negemo", "Tone"]].describe()


,posemo,negemo,Tone
count,234389.000000,234389.000000,234389.000000
mean,5.367127,2.776378,55.067234
std,6.121163,4.480204,40.044771
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,25.770000
50%,4.260000,0.000000,63.690000
75%,7.690000,4.350000,99.000000
max,100.000000,100.000000,99.000000


## 7. Clean critic names

I clean the critic names before creating the CRITICS table.

In [19]:
# JP: I remove extra spaces from critic names and change 'None' into a missing value.
expert["reviewer"] = expert["reviewer"].astype("string").str.strip().replace("None", pd.NA)


## 8. Clean review dates

The original dates are stored as text, so I convert them into proper dates.

In [20]:
# JP: I convert the review dates from text into proper dates for the database.
expert["review_date"] = pd.to_datetime(
    expert["dateP"].astype(str).str.strip().str.strip("'").replace("None", pd.NA),
    format="%b %d, %Y",
    errors="coerce"
)


## 9. Keep the columns needed for the database

The original dataset has many LIWC columns, but our ERD only needs the columns below.

In [21]:
# JP: I keep only the columns needed for the CRITICS and CRITIC_REVIEWS tables.
expert = expert[
    [
        "url",
        "reviewer",
        "review_date",
        "idvscore",
        "Rev",
        "WC",
        "posemo",
        "negemo",
        "Tone"
    ]
]


In [22]:
# JP: I remove extra spaces and quotation marks from the review text.
expert["Rev"] = expert["Rev"].astype("string").str.strip().str.strip("'").str.strip()


## 10. Create the CRITICS table

Each critic should appear only once in the CRITICS table.

In [23]:
# JP: I create the CRITICS table by keeping each critic name only once.
critics = expert[["reviewer"]].dropna().drop_duplicates()


In [24]:
# JP: I give each critic a unique critic_id so the database can identify them.
critics["critic_id"] = range(1, len(critics) + 1)


In [25]:
# JP: I put critic_id first so the table matches the structure of our ERD.
critics = critics[["critic_id", "reviewer"]]


## 11. Connect reviews to critics

I use the critic name to add the critic_id to each review.

In [26]:
# JP: I connect each review to its critic by matching the reviewer name.
expert = expert.merge(critics, on="reviewer", how="left")


## 12. Connect reviews to movies

The reviews contain the movie URL, while our database uses movie_id.

In [27]:
# JP: I load the movies table because I need movie_id for each critic review.
movies = pd.read_csv("movies.csv")


In [28]:
# JP: I connect each review to a movie by matching the movie URL.
expert = expert.merge(movies[["movie_id", "url"]], on="url", how="left")


## 13. Create the CRITIC_REVIEWS table

I keep the review information required for our ERD.

In [29]:
# JP: I create the CRITIC_REVIEWS table using the columns required by our ERD.
critic_reviews = expert[
    [
        "movie_id",
        "critic_id",
        "review_date",
        "idvscore",
        "Rev",
        "WC",
        "posemo",
        "negemo",
        "Tone"
    ]
].copy()


In [30]:
# JP: I rename the columns so their names match the database design.
critic_reviews = critic_reviews.rename(
    columns={
        "Rev": "rev",
        "WC": "wc",
        "Tone": "tone"
    }
)


In [31]:
# JP: I give every critic review a unique critic_review_id.
critic_reviews.insert(0, "critic_review_id", range(1, len(critic_reviews) + 1))


## 14. Save the two tables

I save the finished tables as CSV files so they can be imported into the database.

In [32]:
# JP: I save the CRITICS table as a CSV file.
critics.to_csv("critics.csv", index=False)


In [33]:
# JP: I save the CRITIC_REVIEWS table as a CSV file.
critic_reviews.to_csv("critic_reviews.csv", index=False)


## 15. Final checks

I do a few simple checks to make sure the tables were created correctly.

In [34]:
# JP: I check the final size of the two tables.
print("CRITICS:", critics.shape)
print("CRITIC_REVIEWS:", critic_reviews.shape)


CRITICS: (1178, 2)
CRITIC_REVIEWS: (234389, 10)


In [35]:
# JP: I check that every critic and review has a unique ID.
print("Unique critic IDs:", critics["critic_id"].is_unique)
print("Unique review IDs:", critic_reviews["critic_review_id"].is_unique)


Unique critic IDs: True
Unique review IDs: True


In [36]:
# JP: I check whether any reviews failed to connect to a movie or critic.
print("Missing movie IDs:", critic_reviews["movie_id"].isna().sum())
print("Missing critic IDs:", critic_reviews["critic_id"].isna().sum())


Missing movie IDs: 0
Missing critic IDs: 24879
